# Personal Finance Analysis

**Project 1 — Pandas data analysis only**

This project analyzes a personal-finance transaction dataset using Pandas.
The focus is on cleaning, filtering, grouping, aggregation, date analysis, and summary statistics.

**Dataset:** `personal_finance.csv`
**Rows:** 1,800
**Columns:** 9


In [ ]:
import pandas as pd

df = pd.read_csv("05_personal_finance.csv")
df.head()


## 1. Initial Data Inspection

In [ ]:
df.shape

In [ ]:
df.info()

In [ ]:
df.describe()

In [ ]:
df.dtypes

In [ ]:
df.isna().sum()

In [ ]:
df.nunique()

## 2. Prepare the Date Column

In [ ]:
df["date"] = pd.to_datetime(df["date"])
df["date"].min(), df["date"].max()


## 3. Transaction Summary

In [ ]:
total_amount = df["amount_inr"].sum()
average_transaction = df["amount_inr"].mean()
minimum_transaction = df["amount_inr"].min()
maximum_transaction = df["amount_inr"].max()

print(f"Total transaction amount: ₹{total_amount:,.2f}")
print(f"Average transaction amount: ₹{average_transaction:,.2f}")
print(f"Smallest transaction: ₹{minimum_transaction:,.2f}")
print(f"Largest transaction: ₹{maximum_transaction:,.2f}")


In [ ]:
smallest_transaction = df.loc[df["amount_inr"].idxmin()]
largest_transaction = df.loc[df["amount_inr"].idxmax()]

print("Smallest transaction:")
print(smallest_transaction)
print("\nLargest transaction:")
print(largest_transaction)


## 4. Income vs Expense

In [ ]:
income = df.loc[df["transaction_type"] == "Income"]
expenses = df.loc[df["transaction_type"] == "Expense"]

total_income = income["amount_inr"].sum()
total_expense = expenses["amount_inr"].sum()
net_cash_flow = total_income - total_expense

print(f"Total income: ₹{total_income:,.2f}")
print(f"Total expenses: ₹{total_expense:,.2f}")
print(f"Net cash flow: ₹{net_cash_flow:,.2f}")

if net_cash_flow > 0:
    print("Result: income is higher than expenses.")
elif net_cash_flow < 0:
    print("Result: expenses are higher than income.")
else:
    print("Result: income and expenses are equal.")


## 5. Payment Channel Analysis

In [ ]:
payment_channel_counts = (
    df["payment_channel"]
    .value_counts()
    .rename_axis("payment_channel")
    .reset_index(name="transaction_count")
)

payment_channel_counts


## 6. Transaction Counts by Category

In [ ]:
category_counts = (
    df.groupby(["category", "essential"])
      .size()
      .reset_index(name="transaction_count")
      .sort_values("transaction_count", ascending=False)
)

category_counts


In [ ]:
essential_summary = (
    df.groupby("essential")
      .agg(
          transaction_count=("transaction_id", "count"),
          total_amount=("amount_inr", "sum"),
          average_amount=("amount_inr", "mean")
      )
      .sort_values("total_amount", ascending=False)
)

essential_summary


## 7. Large Transactions

In [ ]:
large_transactions = (
    df.loc[df["amount_inr"] > 11000]
      .sort_values("amount_inr", ascending=False)
)

large_transactions.head(10)


In [ ]:
large_transaction_summary = (
    large_transactions.groupby("transaction_type")
    .agg(
        transaction_count=("transaction_id", "count"),
        total_amount=("amount_inr", "sum"),
        average_amount=("amount_inr", "mean")
    )
)

large_transaction_summary


## 8. Essential vs Non-Essential Large Transactions

In [ ]:
large_transaction_essential = (
    large_transactions.groupby(["transaction_type", "essential"])
    .agg(
        transaction_count=("transaction_id", "count"),
        total_amount=("amount_inr", "sum")
    )
    .sort_values("total_amount", ascending=False)
)

large_transaction_essential


## 9. Monthly Spending

In [ ]:
expenses = df.loc[df["transaction_type"] == "Expense"].copy()
expenses["month"] = expenses["date"].dt.to_period("M")

monthly_spending = (
    expenses.groupby("month")["amount_inr"]
    .sum()
    .sort_index()
)

monthly_spending


In [ ]:
monthly_trend = pd.DataFrame({
    "monthly_spending": monthly_spending,
    "rolling_3_month_average": monthly_spending.rolling(3).mean()
})

monthly_trend


## 10. Category Spending

In [ ]:
category_spending = (
    expenses.groupby("category")
    .agg(
        transaction_count=("transaction_id", "count"),
        total_spending=("amount_inr", "sum"),
        average_transaction=("amount_inr", "mean")
    )
    .sort_values("total_spending", ascending=False)
)

category_spending


## 11. City-Level Analysis

In [ ]:
city_analysis = (
    df.groupby("city")
    .agg(
        transaction_count=("transaction_id", "count"),
        total_amount=("amount_inr", "sum"),
        average_transaction=("amount_inr", "mean")
    )
    .sort_values("total_amount", ascending=False)
)

city_analysis


## 12. Useful Pandas Operations

In [ ]:
top_5_transactions = df.nlargest(5, "amount_inr")
top_5_expenses = expenses.nlargest(5, "amount_inr")
bottom_5_expenses = expenses.nsmallest(5, "amount_inr")

top_5_transactions[["transaction_id", "date", "transaction_type", "category", "amount_inr"]]


In [ ]:
category_transaction_pivot = pd.pivot_table(
    df,
    index="category",
    columns="transaction_type",
    values="amount_inr",
    aggfunc=["count", "sum"],
    fill_value=0
)

category_transaction_pivot


## Conclusion

This project covers the core Pandas workflow used in practical data analysis:

- loading and inspecting data
- checking missing values and unique values
- converting dates
- filtering with boolean conditions
- `groupby()` and `agg()`
- `value_counts()`
- `sort_values()`, `nlargest()`, and `nsmallest()`
- `pivot_table()`
- monthly aggregation with `Period`
- rolling averages
- derived business-style metrics such as net cash flow


